#### Environment Check

In [1]:
# Find the lab folder whether this notebook is run from notebooks/ or retrieval-lab/.
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    LAB_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name == "retrieval-lab":
    LAB_DIR = PROJECT_DIR
else:
    LAB_DIR = PROJECT_DIR / "06-best-practices" / "retrieval-lab"

CODE_DIR = LAB_DIR / "code"

sys.path.append(str(CODE_DIR))

print("Python:", sys.executable)
print("Lab dir:", LAB_DIR)
print("Code dir:", CODE_DIR)

Python: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python
Lab dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/06-best-practices/retrieval-lab
Code dir: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/06-best-practices/retrieval-lab/code


#### Load Search Helpers

In [2]:
# Load the shared search helpers and create reusable clients.
from elasticsearch import ConnectionError

from search import (
    create_es_client,
    create_embedding_model,
    keyword_search,
    vector_search,
    hybrid_search,
)

es_client = create_es_client()

try:
    es_info = es_client.info()
except ConnectionError as error:
    raise RuntimeError(
        "Elasticsearch is not running. From retrieval-lab, run: "
        "docker compose up -d"
    ) from error

embedding_model = create_embedding_model()

es_info


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ObjectApiResponse({'name': '8ccc60c42436', 'cluster_name': 'docker-cluster', 'cluster_uuid': '4CqkjHGoS-eKEv8p4kl9tQ', 'version': {'number': '8.19.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1fde05a4d63448377eceb8fd3d51ce16ca3f02a9', 'build_date': '2025-08-26T02:35:34.366492370Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

#### Check Elasticsearch Index

In [3]:
# This index must exist before search can work. If it is missing, run code/ingest.py.
INDEX_NAME = "course-questions"

index_exists = es_client.indices.exists(index=INDEX_NAME)

print("Index exists:", index_exists)

if not index_exists:
    raise RuntimeError(
        "The Elasticsearch index is missing. From retrieval-lab, run: "
        "uv run python code/ingest.py"
    )


Index exists: True


#### Keyword Search

In [4]:
# Keyword search is strongest when the user query shares exact words with the documents.
query = "I just discovered the course. Can I still join?"

keyword_results = keyword_search(
    query=query,
    num_results=5,
    es_client=es_client,
)

for doc in keyword_results:
    print(doc["course"], "-", doc["question"])

data-engineering-zoomcamp - Course: Can I still join the course after the start date?
data-engineering-zoomcamp - Course: What can I do before the course starts?
data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?
data-engineering-zoomcamp - Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?


#### Vector Search

In [5]:
# Vector search uses embeddings to find documents with similar meaning.
vector_results = vector_search(
    query=query,
    num_results=5,
    es_client=es_client,
    embedding_model=embedding_model,
)

for doc in vector_results:
    print(doc["course"], "-", doc["question"])

data-engineering-zoomcamp - Course: Can I still join the course after the start date?
data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
data-engineering-zoomcamp - Certificate - Can I follow the course in a self-paced mode and get a certificate?
data-engineering-zoomcamp - How can we contribute to the course?
data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?


#### Hybrid Search

In [6]:
# Hybrid search combines keyword matching and vector similarity.
hybrid_results = hybrid_search(
    query=query,
    num_results=5,
    es_client=es_client,
    embedding_model=embedding_model,
)

for doc in hybrid_results:
    print(doc["course"], "-", doc["question"])

data-engineering-zoomcamp - Course: Can I still join the course after the start date?
data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
data-engineering-zoomcamp - Course: What can I do before the course starts?
data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?
data-engineering-zoomcamp - Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?


#### Compare Results

In [8]:
# Print each method in the same format so the results are easy to compare.
def show_results(title, results):
    print(title)
    print("-" * len(title))

    for i, doc in enumerate(results, start=1):
        print(i, doc["course"], "-", doc["question"])

    print()


show_results("Keyword Search", keyword_results)
show_results("Vector Search", vector_results)
show_results("Hybrid Search", hybrid_results)

Keyword Search
--------------
1 data-engineering-zoomcamp - Course: Can I still join the course after the start date?
2 data-engineering-zoomcamp - Course: What can I do before the course starts?
3 data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
4 data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode?
5 data-engineering-zoomcamp - Course: I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?

Vector Search
-------------
1 data-engineering-zoomcamp - Course: Can I still join the course after the start date?
2 data-engineering-zoomcamp - Course - Can I follow the course after it finishes?
3 data-engineering-zoomcamp - Certificate - Can I follow the course in a self-paced mode and get a certificate?
4 data-engineering-zoomcamp - How can we contribute to the course?
5 data-engineering-zoomcamp - Course: Can I get support if I take the course in the self-paced mode

#### Short Learning Summary

In [9]:
print("""
Keyword search is good when the query uses exact words from the documents.

Vector search is good when the query has similar meaning but different words.

Hybrid search combines both approaches so the retriever has a better chance of finding useful context.
""")


Keyword search is good when the query uses exact words from the documents.

Vector search is good when the query has similar meaning but different words.

Hybrid search combines both approaches so the retriever has a better chance of finding useful context.

